# Notebook 155: ベクトル検索とインデックス

## Vector Search & Indexing

---

### このノートブックの位置づけ

**Embeddings シリーズ** の第6章として、大規模ベクトル集合から高速に類似ベクトルを検索するための技術を学びます。

| 項目 | 内容 |
|------|------|
| 難易度 | ★★★☆☆ |
| 所要時間 | 90〜120分 |
| カテゴリ | Embeddings |

### 学習目標

- [ ] 全探索（Brute Force）の計算コストと限界を理解できる
- [ ] FAISSライブラリを使って効率的な近似最近傍探索を実装できる
- [ ] IVF・HNSW・PQ等のインデックス構造の原理を説明できる
- [ ] 検索速度と精度のトレードオフを定量的に評価できる
- [ ] 実用的なベクトル検索パイプラインを構築できる

### 前提知識

- ✅ Notebook 150（埋め込みの幾何学 — 距離と類似度）
- ✅ Notebook 153（文埋め込み）があると望ましい

---

## 目次

1. [環境セットアップ](#1-環境セットアップ)
2. [なぜベクトル検索が重要か？](#2-なぜベクトル検索が重要か)
3. [Brute Force — 全探索の基準線](#3-brute-force--全探索の基準線)
4. [IVF（Inverted File Index）](#4-ivfinverted-file-index)
5. [PQ（Product Quantization）— メモリ効率化](#5-pqproduct-quantization-メモリ効率化)
6. [HNSW（Hierarchical Navigable Small World）](#6-hnswhierarchical-navigable-small-world)
7. [インデックス手法の総合比較](#7-インデックス手法の総合比較)
8. [実践的なベクトル検索パイプライン](#8-実践的なベクトル検索パイプライン)
9. [まとめ・チートシート・よくある間違い・自己評価クイズ](#9-まとめチートシートよくある間違い自己評価クイズ)

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# Section 1: 環境セットアップ
# ============================================================
# 必要なライブラリ:
#   pip install faiss-cpu numpy matplotlib seaborn
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
import sys
import warnings
warnings.filterwarnings('ignore')

# FAISS — Facebook AI Similarity Search
import faiss

# ------------------------------------------------------------
# 日本語フォント設定（環境に応じて調整）
# ------------------------------------------------------------
def setup_japanese_font():
    """日本語フォントをmatplotlibに設定する関数"""
    # macOS: Hiragino Sans, Windows: MS Gothic, Linux: IPAGothic
    font_candidates = ['Hiragino Sans', 'Arial Unicode MS', 'IPAGothic', 'MS Gothic', 'sans-serif']
    plt.rcParams['font.family'] = font_candidates
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (10, 6)
    plt.rcParams['font.size'] = 11
    print("✅ 日本語フォント設定完了")

setup_japanese_font()

# ------------------------------------------------------------
# 再現性のためのシード
# ------------------------------------------------------------
np.random.seed(42)

# Seaborn のスタイル設定
sns.set_style('whitegrid')

print("✅ 環境セットアップ完了")
print(f"   NumPy version: {np.__version__}")
print(f"   FAISS version: {faiss.__version__ if hasattr(faiss, '__version__') else 'N/A'}")
print(f"   Python version: {sys.version.split()[0]}")

---

## 2. なぜベクトル検索が重要か？

### 2.1 ベクトル検索のユースケース

現代のAIシステムでは、テキスト・画像・音声などを **ベクトル（埋め込み）** に変換し、
そのベクトル空間上で「類似するもの」を検索する処理が中核を成しています。

| ユースケース | 説明 |
|-------------|------|
| RAG (Retrieval-Augmented Generation) | LLM に関連文書を検索して渡す |
| 推薦システム | ユーザーの好みに似た商品を検索 |
| 画像検索 | 類似画像の検索 |
| 重複検出 | 類似コンテンツの検出 |
| 意味検索 | キーワードではなく意味で検索 |

### 2.2 規模の問題

実際のアプリケーションでは:
- **N = 100万〜10億** のベクトル
- **D = 768〜1536** 次元（BERTやOpenAIの埋め込み次元）

Brute Force（全探索）の計算量: **O(N × D)** per query

```
N = 1,000,000 × D = 768 = 約7.7億回の浮動小数点演算/クエリ
```

毎秒1000クエリを処理するには？ → **近似最近傍探索（ANN）が必須**

### 2.3 ベクトル検索パイプライン

```
┌─────────────┐     ┌───────────────────────┐     ┌──────────────┐
│  クエリ      │     │   インデックス         │     │  Top-K 結果   │
│  (ベクトル)  │ ──→ │  (ANN データ構造)      │ ──→ │  (類似ベクトル)│
│  D次元       │     │  N個のベクトルを格納    │     │  K個を返す    │
└─────────────┘     └───────────────────────┘     └──────────────┘
                           ↑
                     事前にtrain/buildしておく
```

### 2.4 本ノートブックで学ぶ手法

| 手法 | カテゴリ | 特徴 |
|------|----------|------|
| Flat (Brute Force) | 厳密探索 | 100%正確だが遅い |
| IVF (Inverted File) | 分割ベース | クラスタで候補を絞る |
| PQ (Product Quantization) | 圧縮ベース | メモリ効率が高い |
| HNSW (Hierarchical NSW) | グラフベース | 高速・高精度 |

---

## 3. Brute Force — 全探索の基準線

In [ ]:
# ============================================================
# Section 3: Brute Force — 全探索の基準線
# ============================================================
# まず、NumPy でナイーブなk-NNを実装し、
# FAISSのFlat（厳密探索）と比較する。
# ============================================================

# --- 3.1 NumPy による全探索 ---
def brute_force_knn_numpy(query, data, k=10):
    """
    NumPy でコサイン類似度ベースのk-NNを実装する。
    
    Parameters:
        query: クエリベクトル (D,)
        data:  データベクトル群 (N, D)
        k:     返す近傍数
    Returns:
        indices: 上位k件のインデックス
        similarities: 上位k件のコサイン類似度
    """
    # クエリとデータの正規化
    query_norm = query / np.linalg.norm(query)
    data_norms = np.linalg.norm(data, axis=1, keepdims=True)
    data_normed = data / data_norms
    
    # コサイン類似度 = 正規化ベクトルの内積
    similarities = data_normed @ query_norm
    
    # 上位k件を取得
    top_k_indices = np.argsort(similarities)[::-1][:k]
    top_k_sims = similarities[top_k_indices]
    
    return top_k_indices, top_k_sims


# --- テストデータ生成 ---
N = 100_000  # データベースサイズ
D = 128      # ベクトル次元
K = 10       # 検索する近傍数

# ランダムなベクトルを生成（float32 が FAISS の要件）
np.random.seed(42)
database = np.random.randn(N, D).astype('float32')
queries = np.random.randn(5, D).astype('float32')  # 5個のクエリ

print(f"データベース: {N:,} ベクトル × {D} 次元")
print(f"メモリ使用量: {database.nbytes / 1024**2:.1f} MB")
print(f"クエリ数: {queries.shape[0]}")
print(f"検索近傍数: K={K}")

In [ ]:
# ============================================================
# NumPy vs FAISS Flat の速度比較
# ============================================================

# --- NumPy Brute Force ---
start = time.time()
for q in queries:
    indices_np, sims_np = brute_force_knn_numpy(q, database, k=K)
numpy_time = time.time() - start
print(f"NumPy Brute Force: {numpy_time:.4f} 秒 ({len(queries)} クエリ)")

# --- FAISS IndexFlatL2 ---
# L2距離（ユークリッド距離の2乗）での全探索
index_flat_l2 = faiss.IndexFlatL2(D)
index_flat_l2.add(database)  # データベースを追加
print(f"\nFAISS IndexFlatL2: {index_flat_l2.ntotal:,} ベクトルを格納")

start = time.time()
distances_l2, indices_l2 = index_flat_l2.search(queries, K)
faiss_l2_time = time.time() - start
print(f"FAISS IndexFlatL2: {faiss_l2_time:.4f} 秒 ({len(queries)} クエリ)")

# --- FAISS IndexFlatIP ---
# 内積（Inner Product）での全探索
# コサイン類似度を使うには、ベクトルを事前に正規化する
database_normed = database / np.linalg.norm(database, axis=1, keepdims=True)
queries_normed = queries / np.linalg.norm(queries, axis=1, keepdims=True)

index_flat_ip = faiss.IndexFlatIP(D)
index_flat_ip.add(database_normed)

start = time.time()
similarities_ip, indices_ip = index_flat_ip.search(queries_normed, K)
faiss_ip_time = time.time() - start
print(f"FAISS IndexFlatIP: {faiss_ip_time:.4f} 秒 ({len(queries)} クエリ)")

# --- 速度比較 ---
print(f"\n--- 速度比較 ---")
print(f"NumPy      : {numpy_time:.4f} 秒")
print(f"FAISS L2   : {faiss_l2_time:.4f} 秒 (NumPyの {numpy_time/faiss_l2_time:.1f}倍速)")
print(f"FAISS IP   : {faiss_ip_time:.4f} 秒 (NumPyの {numpy_time/faiss_ip_time:.1f}倍速)")

In [ ]:
# ============================================================
# Plot 1: データサイズ vs 検索時間（Brute Force の線形増加）
# ============================================================

# 異なるデータサイズで検索時間を計測
data_sizes = [1000, 5000, 10000, 25000, 50000, 100000]
search_times_numpy = []
search_times_faiss = []

# 固定のクエリ（1つ）
single_query = queries[:1]  # shape: (1, D)

for n in data_sizes:
    # データのサブセットを取得
    subset = database[:n]
    
    # NumPy での計測
    start = time.time()
    for _ in range(10):  # 10回繰り返して平均
        brute_force_knn_numpy(single_query[0], subset, k=K)
    numpy_t = (time.time() - start) / 10
    search_times_numpy.append(numpy_t * 1000)  # ミリ秒
    
    # FAISS での計測
    idx = faiss.IndexFlatL2(D)
    idx.add(subset)
    start = time.time()
    for _ in range(10):
        idx.search(single_query, K)
    faiss_t = (time.time() - start) / 10
    search_times_faiss.append(faiss_t * 1000)  # ミリ秒

# --- 可視化 ---
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(data_sizes, search_times_numpy, 'o-', linewidth=2, markersize=8,
        color='#e74c3c', label='NumPy Brute Force')
ax.plot(data_sizes, search_times_faiss, 's-', linewidth=2, markersize=8,
        color='#3498db', label='FAISS IndexFlatL2')

ax.set_xlabel('データベースサイズ N', fontsize=12)
ax.set_ylabel('検索時間 [ms]', fontsize=12)
ax.set_title('Plot 1: Brute Force の検索時間はデータサイズに線形比例\n（D=128, K=10, 1クエリあたり）', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# データサイズのフォーマット
ax.set_xticks(data_sizes)
ax.set_xticklabels([f'{n//1000}K' for n in data_sizes])

plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("・Brute Force の検索時間はデータサイズに線形比例 — O(N×D)")
print("・FAISS の Flat インデックスも全探索だが、SIMD最適化により高速")
print("・N=100万、D=768 の場合、1クエリに数百ミリ秒かかる")
print("  → リアルタイム検索にはANN（近似最近傍探索）が必要")

---

## 4. IVF（Inverted File Index）

### 4.1 IVF の原理

IVF は **ボロノイ分割（Voronoi Partition）** に基づく手法です。

1. データベース全体を **nlist 個のクラスタ** に分割（k-means）
2. 各ベクトルを最も近いクラスタに割り当て
3. 検索時: クエリに近い **nprobe 個のクラスタのみ** を探索

```
ボロノイ分割のイメージ（2Dの例）:

  ┌────────────────────────────────┐
  │  ·  ·      ╱    · ·  ·        │
  │    · ★   ╱  ·      ★  ·      │  ★ = クラスタ中心
  │  ·    · ╱     ·  ·    ·       │  · = データ点
  │──────╱──────────────────      │  ╱ = ボロノイ境界
  │  ·  ╱  ·       ·    ·  ·      │
  │   ╱    ★    ·    · ·          │
  │ ╱  ·    ·  ╲     ★    ·      │
  │     ·       ╲  ·    ·         │
  └────────────────────────────────┘

検索時:
  クエリ Q → 最も近い nprobe 個のクラスタを選択
           → そのクラスタ内のベクトルのみと距離計算
           → 計算量: O(nprobe/nlist × N × D) << O(N × D)
```

### 4.2 パラメータ

| パラメータ | 意味 | 典型値 |
|-----------|------|--------|
| nlist | クラスタ数 | sqrt(N) 〜 4*sqrt(N) |
| nprobe | 探索するクラスタ数 | 1 〜 nlist |

- `nprobe = 1`: 最速だが低精度（最近傍がクラスタ境界にいると見逃す）
- `nprobe = nlist`: 全探索と同等（Flat と同じ精度・速度）

In [ ]:
# ============================================================
# Section 4: IVF（Inverted File Index）の実装
# ============================================================

# --- 4.2 FAISS IndexIVFFlat の構築 ---
nlist = 100  # クラスタ数

# 量子化器（各ベクトルをどのクラスタに割り当てるかを決める）
quantizer = faiss.IndexFlatL2(D)

# IVFFlat インデックスの作成
index_ivf = faiss.IndexIVFFlat(quantizer, D, nlist)

# IVF はtrain()が必要（k-meansでクラスタ中心を学習）
print("⏳ IVF インデックスを学習中...")
start = time.time()
index_ivf.train(database)  # クラスタ中心を学習
train_time = time.time() - start
print(f"✅ 学習完了: {train_time:.2f} 秒")

# データベースを追加
start = time.time()
index_ivf.add(database)
add_time = time.time() - start
print(f"✅ データ追加完了: {add_time:.2f} 秒")
print(f"   格納ベクトル数: {index_ivf.ntotal:,}")
print(f"   クラスタ数 (nlist): {nlist}")
print(f"   平均クラスタサイズ: {index_ivf.ntotal // nlist}")

In [ ]:
# ============================================================
# nprobe の影響を調べる（recall と検索時間）
# ============================================================

# まず、厳密解（Flat）を求めておく（正解データ）
# 100個のクエリで評価
np.random.seed(42)
n_eval_queries = 100
eval_queries = np.random.randn(n_eval_queries, D).astype('float32')

# 厳密解
gt_distances, gt_indices = index_flat_l2.search(eval_queries, K)


def compute_recall_at_k(gt_indices, pred_indices, k):
    """
    Recall@K を計算する。
    厳密解の上位K件のうち、予測結果に含まれる割合。
    
    Parameters:
        gt_indices: 厳密解のインデックス (n_queries, K)
        pred_indices: 予測のインデックス (n_queries, K)
        k: 上位K件
    Returns:
        float: 平均 recall@K
    """
    recalls = []
    for i in range(len(gt_indices)):
        gt_set = set(gt_indices[i, :k])
        pred_set = set(pred_indices[i, :k])
        recall = len(gt_set & pred_set) / len(gt_set)
        recalls.append(recall)
    return np.mean(recalls)


# --- nprobe を変化させて recall@10 と検索時間を計測 ---
nprobe_values = [1, 2, 4, 8, 16, 32, 64, 100]
ivf_recalls = []
ivf_times = []

for nprobe in nprobe_values:
    index_ivf.nprobe = nprobe
    
    # 検索時間の計測（3回の平均）
    times = []
    for _ in range(3):
        start = time.time()
        distances, indices = index_ivf.search(eval_queries, K)
        times.append(time.time() - start)
    avg_time = np.mean(times)
    
    # Recall@10 を計算
    recall = compute_recall_at_k(gt_indices, indices, K)
    
    ivf_recalls.append(recall)
    ivf_times.append(avg_time * 1000)  # ミリ秒
    
    print(f"nprobe={nprobe:>3}  |  recall@{K}={recall:.4f}  |  time={avg_time*1000:.2f} ms")

print(f"\n参考: Flat (全探索) の時間 = {faiss_l2_time*1000:.2f} ms")

In [ ]:
# ============================================================
# Plot 2: nprobe vs recall@10 と nprobe vs 検索時間
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左: nprobe vs recall@10 ---
ax = axes[0]
ax.plot(nprobe_values, ivf_recalls, 'o-', linewidth=2, markersize=8,
        color='#2ecc71', label='IVFFlat')
ax.axhline(y=1.0, color='#e74c3c', linestyle='--', linewidth=1.5,
           label='Flat (厳密解)', alpha=0.7)
ax.set_xlabel('nprobe（探索クラスタ数）', fontsize=12)
ax.set_ylabel(f'Recall@{K}', fontsize=12)
ax.set_title('nprobe を増やすと精度が向上', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# --- 右: nprobe vs 検索時間 ---
ax = axes[1]
ax.plot(nprobe_values, ivf_times, 's-', linewidth=2, markersize=8,
        color='#e74c3c', label='IVFFlat')
ax.set_xlabel('nprobe（探索クラスタ数）', fontsize=12)
ax.set_ylabel('検索時間 [ms]', fontsize=12)
ax.set_title('nprobe を増やすと時間も増加', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.suptitle('Plot 2: IVF のトレードオフ — nprobe が精度と速度を制御する', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("【ポイント】")
print("・nprobe=1 では recall が低い（クラスタ境界付近の最近傍を見逃す）")
print("・nprobe を増やすと recall は上がるが、検索時間も増える")
print("・nprobe=nlist (100) では全探索と同じ → recall=1.0")
print("・実運用では nprobe を調整してスイートスポットを探す")

In [ ]:
# ============================================================
# Plot 3: 2D可視化 — ボロノイ領域とクエリの探索範囲
# ============================================================
from scipy.spatial import Voronoi, voronoi_plot_2d

# 2Dデータで IVF の動作を可視化
np.random.seed(42)
n_2d = 2000
nlist_2d = 16
data_2d = np.random.randn(n_2d, 2).astype('float32')

# 2D IVF インデックス構築
quantizer_2d = faiss.IndexFlatL2(2)
index_ivf_2d = faiss.IndexIVFFlat(quantizer_2d, 2, nlist_2d)
index_ivf_2d.train(data_2d)
index_ivf_2d.add(data_2d)

# クラスタ中心を取得
centroids_2d = faiss.vector_to_array(quantizer_2d.xb).reshape(-1, 2)

# クエリポイント
query_2d = np.array([[0.5, 0.3]], dtype='float32')

# クエリに最も近いクラスタを見つける（nprobe=3）
nprobe_vis = 3
# クエリからクラスタ中心への距離を計算
centroid_dists = np.linalg.norm(centroids_2d - query_2d[0], axis=1)
closest_clusters = np.argsort(centroid_dists)[:nprobe_vis]

# 各データ点のクラスタ割り当てを取得
_, assignments = quantizer_2d.search(data_2d, 1)
assignments = assignments.ravel()

# --- 可視化 ---
fig, ax = plt.subplots(figsize=(10, 10))

# ボロノイ図の計算
vor = Voronoi(centroids_2d)

# クラスタごとにデータ点を色分け
colors = plt.cm.tab20(np.linspace(0, 1, nlist_2d))
for cluster_id in range(nlist_2d):
    mask = assignments == cluster_id
    alpha = 0.8 if cluster_id in closest_clusters else 0.15
    size = 20 if cluster_id in closest_clusters else 8
    ax.scatter(data_2d[mask, 0], data_2d[mask, 1],
              c=[colors[cluster_id]], s=size, alpha=alpha)

# ボロノイ境界を描画
voronoi_plot_2d(vor, ax=ax, show_vertices=False, show_points=False,
                line_colors='gray', line_width=1, line_alpha=0.5)

# クラスタ中心をプロット
for i, c in enumerate(centroids_2d):
    marker_color = '#e74c3c' if i in closest_clusters else 'gray'
    marker_size = 150 if i in closest_clusters else 50
    ax.scatter(c[0], c[1], c=marker_color, s=marker_size,
              marker='*', zorder=10, edgecolors='black', linewidth=0.5)

# クエリポイントをプロット
ax.scatter(query_2d[0, 0], query_2d[0, 1], c='#f39c12', s=300,
          marker='X', zorder=15, edgecolors='black', linewidth=2,
          label=f'クエリ Q')

# 探索クラスタのラベル
for i in closest_clusters:
    ax.annotate(f'探索クラスタ {i}',
               centroids_2d[i], textcoords='offset points',
               xytext=(10, 10), fontsize=10, color='#e74c3c',
               fontweight='bold',
               arrowprops=dict(arrowstyle='->', color='#e74c3c'))

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title(f'Plot 3: IVF のボロノイ分割と探索範囲\n'
             f'nlist={nlist_2d}, nprobe={nprobe_vis}（色の濃い点が探索対象）', fontsize=13)
ax.legend(fontsize=11, loc='upper right')
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)

plt.tight_layout()
plt.show()

print(f"【IVF の動作】")
print(f"・全 {n_2d} 点のうち、nprobe={nprobe_vis} クラスタ分のみ探索")
searched = sum(assignments == c for c in closest_clusters)
print(f"・探索した点数: 約 {int(np.sum([np.sum(assignments == c) for c in closest_clusters]))} / {n_2d}")
print(f"  → 計算量を約 {nlist_2d/nprobe_vis:.0f} 分の 1 に削減")

---

## 5. PQ（Product Quantization）— メモリ効率化

### 5.1 PQ の原理

**Product Quantization** は、ベクトルを **サブベクトルに分割** し、
各サブ空間で独立に **量子化（離散化）** することでメモリ使用量を大幅に削減します。

```
Product Quantization の流れ:

元のベクトル (D=128次元):
┌──────────────────────────────────────────────────────┐
│ v₁ v₂ v₃ ... v₃₂ │ v₃₃ v₃₄ ... v₆₄ │ ... │ v₉₇ ... v₁₂₈ │
└──────────────────────────────────────────────────────┘
        ↓ M=4 サブベクトルに分割
┌────────────┐ ┌────────────┐ ┌────────────┐ ┌────────────┐
│ サブ空間 1  │ │ サブ空間 2  │ │ サブ空間 3  │ │ サブ空間 4  │
│  (32次元)   │ │  (32次元)   │ │  (32次元)   │ │  (32次元)   │
└────────────┘ └────────────┘ └────────────┘ └────────────┘
        ↓ 各サブ空間で k-means（256クラスタ）
┌────┐         ┌────┐         ┌────┐         ┌────┐
│ 42 │         │ 187│         │ 3  │         │ 255│  ← コードID (0-255)
└────┘         └────┘         └────┘         └────┘
        ↓ 128次元×4byte = 512 byte → 4 byte に圧縮！
┌──────────────────┐
│ 42, 187, 3, 255  │  ← たった4バイトで128次元ベクトルを表現
└──────────────────┘
```

### 5.2 メモリ削減効果

| 手法 | 1ベクトルのサイズ | N=100万での合計 |
|------|-----------------|----------------|
| Flat (float32) | D × 4 = 512 byte | 512 MB |
| PQ (M=4, 8bit) | M × 1 = 4 byte | 4 MB |
| 圧縮率 | — | **128倍** |

In [ ]:
# ============================================================
# Section 5: PQ（Product Quantization）の実装
# ============================================================

# PQ のパラメータ
M_pq = 16        # サブベクトルの数（D が M_pq で割り切れる必要がある）
nbits = 8        # 各サブ空間のビット数（2^8 = 256 クラスタ）

print(f"PQ パラメータ:")
print(f"  D = {D} 次元")
print(f"  M = {M_pq} サブベクトル")
print(f"  各サブベクトルの次元: {D // M_pq}")
print(f"  nbits = {nbits} → 各サブ空間で {2**nbits} クラスタ")
print(f"  圧縮後のサイズ: {M_pq} byte/ベクトル")
print(f"  圧縮率: {D * 4 / M_pq:.0f}倍")

# --- FAISS IndexIVFPQ の構築 ---
nlist_pq = 100
quantizer_pq = faiss.IndexFlatL2(D)
index_ivfpq = faiss.IndexIVFPQ(quantizer_pq, D, nlist_pq, M_pq, nbits)

# 学習（k-means for IVF + k-means for PQ codebooks）
print(f"\n⏳ IVFPQ インデックスを学習中...")
start = time.time()
index_ivfpq.train(database)
pq_train_time = time.time() - start
print(f"✅ 学習完了: {pq_train_time:.2f} 秒")

# データ追加
start = time.time()
index_ivfpq.add(database)
pq_add_time = time.time() - start
print(f"✅ データ追加完了: {pq_add_time:.2f} 秒")
print(f"   格納ベクトル数: {index_ivfpq.ntotal:,}")

In [ ]:
# ============================================================
# メモリ使用量の比較
# ============================================================

# 各インデックスのメモリ使用量を概算
# Flat: N × D × 4 byte
flat_memory = N * D * 4  # bytes
# IVFFlat: N × D × 4 + nlist × D × 4 (centroids)
ivfflat_memory = N * D * 4 + nlist * D * 4
# IVFPQ: N × M_pq + コードブック + セントロイド
ivfpq_memory = N * M_pq + nlist_pq * D * 4 + M_pq * (2**nbits) * (D // M_pq) * 4

print("=" * 60)
print("メモリ使用量の比較（概算）")
print("=" * 60)
print(f"  Flat       : {flat_memory / 1024**2:.1f} MB")
print(f"  IVFFlat    : {ivfflat_memory / 1024**2:.1f} MB")
print(f"  IVFPQ      : {ivfpq_memory / 1024**2:.1f} MB  ({flat_memory / ivfpq_memory:.0f}倍圧縮)")

# --- PQ の精度評価 ---
index_ivfpq.nprobe = 16
pq_distances, pq_indices = index_ivfpq.search(eval_queries, K)
pq_recall = compute_recall_at_k(gt_indices, pq_indices, K)
print(f"\n  IVFPQ Recall@{K} (nprobe=16): {pq_recall:.4f}")
print(f"  → PQ による量子化誤差があるため、IVFFlat より recall が低い")

In [ ]:
# ============================================================
# Plot 4: 元のベクトル vs PQ復元ベクトルの誤差分布
# ============================================================

# PQ で圧縮→復元したベクトルと元のベクトルの差を分析
# FAISS の IndexPQ を使って復元ベクトルを取得
index_pq_only = faiss.IndexPQ(D, M_pq, nbits)
index_pq_only.train(database)
index_pq_only.add(database)

# 復元ベクトルの取得
# reconstruct で元のベクトルを復元
n_sample = 1000  # サンプル数
sample_indices = np.random.choice(N, n_sample, replace=False)

original_vecs = database[sample_indices]
reconstructed_vecs = np.array([index_pq_only.reconstruct(int(i)) for i in sample_indices])

# 各ベクトルの再構成誤差（ユークリッド距離）
reconstruction_errors = np.linalg.norm(original_vecs - reconstructed_vecs, axis=1)

# 元のベクトルのノルム
original_norms = np.linalg.norm(original_vecs, axis=1)

# 相対誤差
relative_errors = reconstruction_errors / original_norms

# --- 可視化 ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 左: 絶対誤差の分布
ax = axes[0]
ax.hist(reconstruction_errors, bins=50, color='#3498db', edgecolor='black',
        linewidth=0.3, alpha=0.8)
ax.axvline(np.mean(reconstruction_errors), color='#e74c3c', linestyle='--',
           linewidth=2, label=f'平均: {np.mean(reconstruction_errors):.2f}')
ax.set_xlabel('再構成誤差（L2距離）', fontsize=11)
ax.set_ylabel('頻度', fontsize=11)
ax.set_title('PQ 再構成の絶対誤差', fontsize=12)
ax.legend(fontsize=10)

# 中: 相対誤差の分布
ax = axes[1]
ax.hist(relative_errors, bins=50, color='#2ecc71', edgecolor='black',
        linewidth=0.3, alpha=0.8)
ax.axvline(np.mean(relative_errors), color='#e74c3c', linestyle='--',
           linewidth=2, label=f'平均: {np.mean(relative_errors):.3f}')
ax.set_xlabel('相対誤差（L2距離 / ノルム）', fontsize=11)
ax.set_ylabel('頻度', fontsize=11)
ax.set_title('PQ 再構成の相対誤差', fontsize=12)
ax.legend(fontsize=10)

# 右: 元 vs 復元の各次元の比較（1サンプル）
ax = axes[2]
sample_idx = 0
dims_to_show = min(D, 64)
ax.plot(range(dims_to_show), original_vecs[sample_idx, :dims_to_show],
        'o-', markersize=3, linewidth=1, color='#3498db', label='元のベクトル', alpha=0.8)
ax.plot(range(dims_to_show), reconstructed_vecs[sample_idx, :dims_to_show],
        's-', markersize=3, linewidth=1, color='#e74c3c', label='PQ復元ベクトル', alpha=0.8)
ax.set_xlabel('次元', fontsize=11)
ax.set_ylabel('値', fontsize=11)
ax.set_title(f'1ベクトルの各次元比較\n(最初の{dims_to_show}次元)', fontsize=12)
ax.legend(fontsize=10)

plt.suptitle(f'Plot 4: PQ の再構成誤差（M={M_pq}, nbits={nbits}）', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"【PQ の量子化誤差】")
print(f"  平均絶対誤差: {np.mean(reconstruction_errors):.3f}")
print(f"  平均相対誤差: {np.mean(relative_errors):.3f} ({np.mean(relative_errors)*100:.1f}%)")
print(f"  メモリ削減: {D * 4} byte → {M_pq} byte ({D*4/M_pq:.0f}倍圧縮)")
print(f"  → 精度を少し犠牲にして、大幅なメモリ削減を実現")

---

## 6. HNSW（Hierarchical Navigable Small World）

### 6.1 HNSW の原理

HNSW は **グラフベース** の近似最近傍探索手法です。
データ点をノードとする多層グラフを構築し、上位層から下位層へと
段階的に探索を絞り込みます。

```
HNSW の階層構造:

Layer 2 (最上位):  少数のノード、長距離リンク
  ○ ────────────── ○ ──────── ○
  │                 │          │
  ↓                 ↓          ↓
Layer 1 (中間):     中程度のノード、中距離リンク
  ○ ─── ○ ──── ○ ── ○ ─── ○ ── ○
  │     │      │    │     │    │
  ↓     ↓      ↓    ↓     ↓    ↓
Layer 0 (最下位):   全ノード、短距離リンク
  ○─○─○─○─○─○─○─○─○─○─○─○─○─○─○

探索の流れ:
  1. Layer 2 でエントリポイントから最も近いノードへ greedy search
  2. そのノードを起点に Layer 1 で greedy search
  3. Layer 0 で最終的な近傍を探索
  → 「高速道路」(上位層) から「一般道」(下位層) へ
```

### 6.2 パラメータ

| パラメータ | 意味 | 典型値 |
|-----------|------|--------|
| M | 各ノードの接続数（グラフの密度） | 16〜64 |
| efConstruction | 構築時の探索幅 | 40〜200 |
| efSearch | 検索時の探索幅 | 16〜256 |

- **M が大きい**: グラフが密になり精度向上するが、メモリと構築時間が増加
- **efSearch が大きい**: 探索範囲が広がり精度向上するが、検索時間が増加

In [ ]:
# ============================================================
# Section 6: HNSW（Hierarchical Navigable Small World）の実装
# ============================================================

# HNSW パラメータ
M_hnsw = 32           # 各ノードの接続数
ef_construction = 64  # 構築時の探索幅

# FAISS IndexHNSWFlat の構築
print(f"HNSW パラメータ:")
print(f"  M = {M_hnsw}")
print(f"  efConstruction = {ef_construction}")

index_hnsw = faiss.IndexHNSWFlat(D, M_hnsw)
index_hnsw.hnsw.efConstruction = ef_construction

# データ追加（HNSW はtrain不要、add時にグラフを構築）
print(f"\n⏳ HNSW グラフを構築中...")
start = time.time()
index_hnsw.add(database)
hnsw_build_time = time.time() - start
print(f"✅ 構築完了: {hnsw_build_time:.2f} 秒")
print(f"   格納ベクトル数: {index_hnsw.ntotal:,}")

# --- efSearch を変化させて性能を評価 ---
ef_search_values = [4, 8, 16, 32, 64, 128, 256, 512]
hnsw_recalls = []
hnsw_times = []

for ef_search in ef_search_values:
    index_hnsw.hnsw.efSearch = ef_search
    
    # 検索時間の計測
    times = []
    for _ in range(3):
        start = time.time()
        distances, indices = index_hnsw.search(eval_queries, K)
        times.append(time.time() - start)
    avg_time = np.mean(times)
    
    # Recall@10
    recall = compute_recall_at_k(gt_indices, indices, K)
    
    hnsw_recalls.append(recall)
    hnsw_times.append(avg_time * 1000)  # ミリ秒
    
    print(f"efSearch={ef_search:>3}  |  recall@{K}={recall:.4f}  |  time={avg_time*1000:.2f} ms")

In [ ]:
# ============================================================
# Plot 5: M と efSearch が recall と速度に与える影響
# ============================================================

# 異なる M で HNSW を構築して比較
M_values = [8, 16, 32, 64]
ef_search_test = [8, 16, 32, 64, 128, 256]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_m = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for m_idx, M_val in enumerate(M_values):
    # HNSW インデックスの構築
    idx_h = faiss.IndexHNSWFlat(D, M_val)
    idx_h.hnsw.efConstruction = 64
    idx_h.add(database)
    
    recalls_m = []
    times_m = []
    
    for ef_s in ef_search_test:
        idx_h.hnsw.efSearch = ef_s
        
        # 計測
        t_list = []
        for _ in range(3):
            start = time.time()
            dists, idxs = idx_h.search(eval_queries, K)
            t_list.append(time.time() - start)
        avg_t = np.mean(t_list)
        
        rec = compute_recall_at_k(gt_indices, idxs, K)
        recalls_m.append(rec)
        times_m.append(avg_t * 1000)
    
    # 左: efSearch vs recall
    axes[0].plot(ef_search_test, recalls_m, 'o-', linewidth=2, markersize=6,
                color=colors_m[m_idx], label=f'M={M_val}')
    
    # 右: efSearch vs 検索時間
    axes[1].plot(ef_search_test, times_m, 's-', linewidth=2, markersize=6,
                color=colors_m[m_idx], label=f'M={M_val}')

# 左グラフの設定
axes[0].set_xlabel('efSearch', fontsize=12)
axes[0].set_ylabel(f'Recall@{K}', fontsize=12)
axes[0].set_title('efSearch vs Recall', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.05)

# 右グラフの設定
axes[1].set_xlabel('efSearch', fontsize=12)
axes[1].set_ylabel('検索時間 [ms]', fontsize=12)
axes[1].set_title('efSearch vs 検索時間', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Plot 5: HNSW の M と efSearch が性能に与える影響', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("【ポイント】")
print("・M が大きいほど高い recall を達成できるが、メモリと構築時間が増加")
print("・efSearch を増やすと recall は改善するが、検索時間も増加")
print("・HNSW は比較的少ない efSearch でも高い recall を達成できる")
print("  → グラフベース手法の強み")

---

## 7. インデックス手法の総合比較

In [ ]:
# ============================================================
# Section 7: インデックス手法の総合比較
# ============================================================
# 同一データセット（N=100,000, D=128）で全手法を比較する
# 比較軸: 検索時間、recall@10、メモリ使用量、構築時間
# ============================================================

# --- 各手法のインデックスを再構築して統一的に計測 ---

results = {}

# (1) Flat (Brute Force)
print("=" * 60)
print("(1) Flat (Brute Force)")
print("=" * 60)
idx_flat = faiss.IndexFlatL2(D)
start = time.time()
idx_flat.add(database)
flat_build = time.time() - start

# 検索
times_flat = []
for _ in range(5):
    start = time.time()
    d_flat, i_flat = idx_flat.search(eval_queries, K)
    times_flat.append(time.time() - start)
flat_search = np.mean(times_flat) * 1000
flat_recall = 1.0  # 厳密解
flat_mem = N * D * 4 / 1024**2  # MB

results['Flat'] = {
    'search_ms': flat_search,
    'recall': flat_recall,
    'memory_mb': flat_mem,
    'build_s': flat_build,
    'qps': len(eval_queries) / (np.mean(times_flat)),
}
print(f"  構築: {flat_build:.3f}s | 検索: {flat_search:.2f}ms | recall: {flat_recall:.4f} | メモリ: {flat_mem:.1f}MB")

# (2) IVFFlat
print("\n" + "=" * 60)
print("(2) IVFFlat")
print("=" * 60)
nlist_cmp = 100
quantizer_cmp = faiss.IndexFlatL2(D)
idx_ivf_cmp = faiss.IndexIVFFlat(quantizer_cmp, D, nlist_cmp)
start = time.time()
idx_ivf_cmp.train(database)
idx_ivf_cmp.add(database)
ivf_build = time.time() - start
idx_ivf_cmp.nprobe = 16

times_ivf = []
for _ in range(5):
    start = time.time()
    d_ivf, i_ivf = idx_ivf_cmp.search(eval_queries, K)
    times_ivf.append(time.time() - start)
ivf_search = np.mean(times_ivf) * 1000
ivf_recall = compute_recall_at_k(gt_indices, i_ivf, K)
ivf_mem = (N * D * 4 + nlist_cmp * D * 4) / 1024**2

results['IVFFlat'] = {
    'search_ms': ivf_search,
    'recall': ivf_recall,
    'memory_mb': ivf_mem,
    'build_s': ivf_build,
    'qps': len(eval_queries) / (np.mean(times_ivf)),
}
print(f"  構築: {ivf_build:.3f}s | 検索: {ivf_search:.2f}ms | recall: {ivf_recall:.4f} | メモリ: {ivf_mem:.1f}MB")

# (3) IVFPQ
print("\n" + "=" * 60)
print("(3) IVFPQ")
print("=" * 60)
quantizer_pq_cmp = faiss.IndexFlatL2(D)
idx_ivfpq_cmp = faiss.IndexIVFPQ(quantizer_pq_cmp, D, nlist_cmp, M_pq, nbits)
start = time.time()
idx_ivfpq_cmp.train(database)
idx_ivfpq_cmp.add(database)
pq_build = time.time() - start
idx_ivfpq_cmp.nprobe = 16

times_pq = []
for _ in range(5):
    start = time.time()
    d_pq, i_pq = idx_ivfpq_cmp.search(eval_queries, K)
    times_pq.append(time.time() - start)
pq_search = np.mean(times_pq) * 1000
pq_recall = compute_recall_at_k(gt_indices, i_pq, K)
pq_mem_cmp = (N * M_pq + nlist_cmp * D * 4 + M_pq * 256 * (D // M_pq) * 4) / 1024**2

results['IVFPQ'] = {
    'search_ms': pq_search,
    'recall': pq_recall,
    'memory_mb': pq_mem_cmp,
    'build_s': pq_build,
    'qps': len(eval_queries) / (np.mean(times_pq)),
}
print(f"  構築: {pq_build:.3f}s | 検索: {pq_search:.2f}ms | recall: {pq_recall:.4f} | メモリ: {pq_mem_cmp:.1f}MB")

# (4) HNSW
print("\n" + "=" * 60)
print("(4) HNSW")
print("=" * 60)
idx_hnsw_cmp = faiss.IndexHNSWFlat(D, 32)
idx_hnsw_cmp.hnsw.efConstruction = 64
start = time.time()
idx_hnsw_cmp.add(database)
hnsw_build = time.time() - start
idx_hnsw_cmp.hnsw.efSearch = 64

times_hnsw = []
for _ in range(5):
    start = time.time()
    d_hnsw, i_hnsw = idx_hnsw_cmp.search(eval_queries, K)
    times_hnsw.append(time.time() - start)
hnsw_search = np.mean(times_hnsw) * 1000
hnsw_recall = compute_recall_at_k(gt_indices, i_hnsw, K)
# HNSW メモリ: ベクトル + グラフ構造
# グラフ: 各ノード最大 M*2 個の接続 × 4byte (int32)
hnsw_mem = (N * D * 4 + N * 32 * 2 * 4) / 1024**2

results['HNSW'] = {
    'search_ms': hnsw_search,
    'recall': hnsw_recall,
    'memory_mb': hnsw_mem,
    'build_s': hnsw_build,
    'qps': len(eval_queries) / (np.mean(times_hnsw)),
}
print(f"  構築: {hnsw_build:.3f}s | 検索: {hnsw_search:.2f}ms | recall: {hnsw_recall:.4f} | メモリ: {hnsw_mem:.1f}MB")

In [ ]:
# ============================================================
# Plot 6: Recall vs Queries/sec の散布図
# ============================================================

fig, ax = plt.subplots(figsize=(10, 7))

method_colors = {
    'Flat': '#e74c3c',
    'IVFFlat': '#3498db',
    'IVFPQ': '#2ecc71',
    'HNSW': '#9b59b6',
}
method_markers = {
    'Flat': 'o',
    'IVFFlat': 's',
    'IVFPQ': 'D',
    'HNSW': '^',
}

for method, r in results.items():
    ax.scatter(r['recall'], r['qps'], c=method_colors[method],
              marker=method_markers[method], s=200, zorder=5,
              edgecolors='black', linewidth=1, label=method)
    ax.annotate(method, (r['recall'], r['qps']),
               textcoords='offset points', xytext=(10, 10),
               fontsize=12, fontweight='bold', color=method_colors[method])

ax.set_xlabel(f'Recall@{K}', fontsize=13)
ax.set_ylabel('Queries/sec（対数スケール）', fontsize=13)
ax.set_title('Plot 6: Recall vs スループットの比較\n（右上ほど理想的）', fontsize=14)
ax.set_yscale('log')
ax.legend(fontsize=11, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1.05)

# 「理想」方向を示す矢印
ax.annotate('理想', xy=(0.95, ax.get_ylim()[1] * 0.7),
           fontsize=14, fontweight='bold', color='gray',
           ha='center')
ax.annotate('', xy=(0.98, ax.get_ylim()[1] * 0.5),
           xytext=(0.85, ax.get_ylim()[1] * 0.2),
           arrowprops=dict(arrowstyle='->', color='gray', lw=2))

plt.tight_layout()
plt.show()

print("【各手法の特徴】")
print("・Flat: recall=1.0 だがスループットが最も低い")
print("・IVFFlat: Flat と同程度のメモリで高速化")
print("・IVFPQ: メモリ効率が最も高いが recall は低め")
print("・HNSW: 高い recall と高速の両立だがメモリ消費が大きい")

In [ ]:
# ============================================================
# Plot 7: メモリ使用量の棒グラフ比較
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

methods = list(results.keys())
bar_colors = [method_colors[m] for m in methods]

# --- 左: メモリ使用量 ---
ax = axes[0]
memories = [results[m]['memory_mb'] for m in methods]
bars = ax.bar(methods, memories, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('メモリ使用量 [MB]', fontsize=12)
ax.set_title('メモリ使用量', fontsize=13)
for bar, val in zip(bars, memories):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# --- 中: 構築時間 ---
ax = axes[1]
build_times = [results[m]['build_s'] for m in methods]
bars = ax.bar(methods, build_times, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('構築時間 [秒]', fontsize=12)
ax.set_title('インデックス構築時間', fontsize=13)
for bar, val in zip(bars, build_times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# --- 右: 検索時間 ---
ax = axes[2]
search_times = [results[m]['search_ms'] for m in methods]
bars = ax.bar(methods, search_times, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('検索時間 [ms]', fontsize=12)
ax.set_title(f'検索時間（{n_eval_queries}クエリ）', fontsize=13)
for bar, val in zip(bars, search_times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}ms', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Plot 7: インデックス手法の総合比較', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 比較結果のテーブル表示
# ============================================================

print("=" * 80)
print("インデックス手法の総合比較（N=100,000, D=128, K=10）")
print("=" * 80)
print(f"{'手法':<12} {'Recall@10':>10} {'検索時間(ms)':>14} {'QPS':>10} {'メモリ(MB)':>12} {'構築時間(s)':>12}")
print("-" * 80)
for method, r in results.items():
    print(f"{method:<12} {r['recall']:>10.4f} {r['search_ms']:>14.2f} {r['qps']:>10.0f} {r['memory_mb']:>12.1f} {r['build_s']:>12.3f}")
print("-" * 80)

print("\n【選び方のガイドライン】")
print("  - 精度最優先           → Flat（小規模データ向け）")
print("  - バランス型           → IVFFlat（中規模、nprobe で調整可能）")
print("  - メモリ制約が厳しい   → IVFPQ（大規模データ向け）")
print("  - 高速・高精度が必要   → HNSW（メモリに余裕がある場合）")

---

## 8. 実践的なベクトル検索パイプライン

ここでは、実際にテキストをエンコードしてFAISSインデックスで意味検索を行うデモを示します。

> **注意**: `sentence-transformers` がインストールされていない場合は、事前計算済みの
> ダミー埋め込みを使ったデモにフォールバックします。

In [ ]:
# ============================================================
# Section 8: 実践的なベクトル検索パイプライン
# ============================================================

# --- 8.1 テキストデータの準備 ---
# サンプル文書（日本語・英語混在）
documents = [
    "Python is a popular programming language for machine learning.",
    "Deep learning requires large datasets and GPU computation.",
    "Natural language processing deals with text and speech.",
    "Computer vision focuses on image recognition and object detection.",
    "Reinforcement learning is used in robotics and game playing.",
    "Transfer learning reduces the need for large labeled datasets.",
    "BERT is a transformer model for natural language understanding.",
    "Convolutional neural networks are effective for image classification.",
    "Recurrent neural networks can process sequential data.",
    "Generative adversarial networks create realistic synthetic data.",
    "Attention mechanisms are key components of transformer architectures.",
    "Feature engineering is important for traditional machine learning.",
    "Dimensionality reduction helps visualize high-dimensional data.",
    "Clustering algorithms group similar data points together.",
    "Gradient descent is the most common optimization algorithm.",
    "Overfitting occurs when a model memorizes training data.",
    "Cross-validation helps evaluate model generalization.",
    "Random forests combine multiple decision trees.",
    "Support vector machines find optimal decision boundaries.",
    "K-nearest neighbors is a simple but effective algorithm.",
]

print(f"文書数: {len(documents)}")
for i, doc in enumerate(documents[:5]):
    print(f"  [{i}] {doc}")
print(f"  ...（残り {len(documents)-5} 文書）")

In [ ]:
# ============================================================
# 8.2 文のエンコードと FAISS インデックス構築
# ============================================================

try:
    # sentence-transformers が利用可能な場合
    from sentence_transformers import SentenceTransformer
    
    print("⏳ SentenceTransformer モデルをロード中...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # 文書をエンコード
    print("⏳ 文書をエンコード中...")
    doc_embeddings = model.encode(documents, show_progress_bar=False)
    doc_embeddings = doc_embeddings.astype('float32')
    
    use_real_model = True
    print(f"✅ エンコード完了: {doc_embeddings.shape}")
    
except ImportError:
    # sentence-transformers がない場合はダミー埋め込みを使用
    print("⚠️ sentence-transformers が未インストール")
    print("   ダミー埋め込み（ランダムベクトル）を使用します")
    print("   インストール: pip install sentence-transformers")
    
    np.random.seed(123)
    embed_dim = 384  # all-MiniLM-L6-v2 の次元に合わせる
    doc_embeddings = np.random.randn(len(documents), embed_dim).astype('float32')
    # 正規化
    doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    
    use_real_model = False
    print(f"✅ ダミー埋め込み生成完了: {doc_embeddings.shape}")

# --- FAISS インデックスの構築 ---
embed_dim = doc_embeddings.shape[1]

# 正規化してコサイン類似度を内積で計算できるようにする
faiss.normalize_L2(doc_embeddings)

# IndexFlatIP（内積 = コサイン類似度）
search_index = faiss.IndexFlatIP(embed_dim)
search_index.add(doc_embeddings)

print(f"\n✅ FAISS インデックス構築完了")
print(f"   格納ベクトル数: {search_index.ntotal}")
print(f"   次元: {embed_dim}")

In [ ]:
# ============================================================
# 8.3 意味検索デモ
# ============================================================

def semantic_search(query_text, index, docs, top_k=5):
    """
    意味検索を実行する。
    
    Parameters:
        query_text: 検索クエリ（テキスト）
        index: FAISS インデックス
        docs: 文書リスト
        top_k: 返す結果数
    """
    # クエリのエンコード
    if use_real_model:
        query_vec = model.encode([query_text]).astype('float32')
    else:
        # ダミーモードでは固定のランダムベクトル
        np.random.seed(hash(query_text) % 2**32)
        query_vec = np.random.randn(1, embed_dim).astype('float32')
    
    faiss.normalize_L2(query_vec)
    
    # 検索実行
    similarities, indices = index.search(query_vec, top_k)
    
    print(f"\n検索クエリ: '{query_text}'")
    print(f"{'-' * 60}")
    for rank, (idx, sim) in enumerate(zip(indices[0], similarities[0]), 1):
        bar = '#' * int(sim * 30) if sim > 0 else ''
        print(f"  {rank}. [{sim:.4f}] {docs[idx]}")
        print(f"     {bar}")


# --- 検索テスト ---
test_queries = [
    "How to process text data?",
    "image recognition with neural networks",
    "preventing model overfitting",
]

for query in test_queries:
    semantic_search(query, search_index, documents, top_k=3)

if not use_real_model:
    print("\n⚠️ ダミー埋め込みのため、検索結果は意味的に正確ではありません")
    print("   実際の意味検索を試すには: pip install sentence-transformers")

In [ ]:
# ============================================================
# 8.4 実運用での考慮事項
# ============================================================

print("=" * 70)
print("実運用でのベクトル検索 — 考慮すべきポイント")
print("=" * 70)

# --- (a) インデックスの保存と読み込み ---
print("\n【1. インデックスの保存と読み込み】")
print("-" * 50)

# 保存
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    index_path = os.path.join(tmpdir, "my_index.faiss")
    
    # 保存
    faiss.write_index(search_index, index_path)
    file_size = os.path.getsize(index_path)
    print(f"  保存: faiss.write_index(index, '{index_path}')")
    print(f"  ファイルサイズ: {file_size / 1024:.1f} KB")
    
    # 読み込み
    loaded_index = faiss.read_index(index_path)
    print(f"  読み込み: faiss.read_index('{index_path}')")
    print(f"  復元ベクトル数: {loaded_index.ntotal}")
    
    # 検証: 同じ結果が得られるか
    test_q = doc_embeddings[:1]
    d1, i1 = search_index.search(test_q, 3)
    d2, i2 = loaded_index.search(test_q, 3)
    print(f"  検証: 保存前後で同一結果 = {np.array_equal(i1, i2)}")

# --- (b) GPU での高速化 ---
print("\n【2. GPU での高速化】")
print("-" * 50)
print("  # GPU リソースの作成")
print("  res = faiss.StandardGpuResources()")
print("  ")
print("  # CPU インデックスを GPU に転送")
print("  gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)")
print("  ")
print("  # 検索（API は同じ）")
print("  distances, indices = gpu_index.search(queries, K)")
print("  ")
print("  # 複数GPU の場合")
print("  gpu_index = faiss.index_cpu_to_all_gpus(cpu_index)")
print("  ")
print("  ※ faiss-gpu パッケージが必要")

# --- (c) バッチ検索 ---
print("\n【3. バッチ検索】")
print("-" * 50)

# シングル vs バッチの速度比較
n_batch_queries = 100
batch_queries = np.random.randn(n_batch_queries, D).astype('float32')

# シングル検索
start = time.time()
for q in batch_queries:
    idx_flat.search(q.reshape(1, -1), K)
single_time = time.time() - start

# バッチ検索
start = time.time()
idx_flat.search(batch_queries, K)
batch_time = time.time() - start

print(f"  シングル検索 ({n_batch_queries}回): {single_time*1000:.2f} ms")
print(f"  バッチ検索   ({n_batch_queries}件): {batch_time*1000:.2f} ms")
print(f"  高速化: {single_time/batch_time:.1f}倍")
print(f"  → バッチ検索は SIMD 並列化の恩恵を最大限受ける")

# --- (d) フィルタリングとの組み合わせ ---
print("\n【4. フィルタリングとの組み合わせ】")
print("-" * 50)
print("  実運用では『ベクトル検索 + メタデータフィルタ』が一般的:")
print("  ")
print("  方式1: Pre-filtering（検索前にフィルタ）")
print("    → IDSelectorArray でフィルタ済みIDを指定")
print("  ")
print("  方式2: Post-filtering（検索後にフィルタ）")
print("    → K を大きめに取って検索 → フィルタ → 上位K件を返す")
print("  ")
print("  方式3: 専用ベクトルDB（Pinecone, Weaviate, Qdrant等）")
print("    → メタデータフィルタとベクトル検索を統合したAPI")

---

## 9. まとめ・チートシート・よくある間違い・自己評価クイズ

### 9.1 まとめ

このノートブックでは、大規模ベクトル集合に対する効率的な検索手法を学びました。

1. **Brute Force**: 厳密だが O(N×D) で大規模データには不向き
2. **IVF**: ボロノイ分割で候補を絞り込み、nprobe で精度と速度をトレードオフ
3. **PQ**: ベクトルを圧縮して大幅にメモリ削減（数十〜百倍の圧縮率）
4. **HNSW**: グラフ構造による高速・高精度な探索（ただしメモリ消費が大きい）
5. **実践**: FAISSを使ったインデックス構築、保存/読み込み、バッチ検索

### 9.2 チートシート

| 手法 | 速度 | 精度 | メモリ | 構築時間 | 適用場面 |
|------|------|------|--------|----------|----------|
| Flat | 遅い | 100% | 大 | 瞬時 | 小規模データ（〜10万） |
| IVFFlat | 中 | 高い | 大 | 中 | 中規模（10万〜1000万） |
| IVFPQ | 速い | 中程度 | 小 | 長い | 大規模（100万〜10億） |
| HNSW | 速い | 高い | 大 | 長い | 高精度が必要な場合 |

**FAISS の主要 API:**

```python
# インデックス作成
index = faiss.IndexFlatL2(d)           # L2 全探索
index = faiss.IndexFlatIP(d)           # 内積 全探索
index = faiss.IndexIVFFlat(quantizer, d, nlist)  # IVF
index = faiss.IndexIVFPQ(quantizer, d, nlist, M, nbits)  # IVFPQ
index = faiss.IndexHNSWFlat(d, M)      # HNSW

# 操作
index.train(data)                      # 学習（IVF系のみ）
index.add(data)                        # データ追加
D, I = index.search(queries, k)        # 検索

# 保存/読み込み
faiss.write_index(index, "path.faiss")
index = faiss.read_index("path.faiss")
```

### 9.3 よくある間違い

#### 間違い 1: L2距離とコサイン類似度を混同する（正規化の重要性）

FAISS の `IndexFlatL2` は **ユークリッド距離** を使います。コサイン類似度で検索したい場合は、ベクトルを **L2正規化** してから `IndexFlatIP`（内積）を使う必要があります。

```python
# 正しい方法:
faiss.normalize_L2(vectors)         # L2正規化
index = faiss.IndexFlatIP(d)        # 内積 = コサイン類似度（正規化後）

# よくある間違い:
index = faiss.IndexFlatL2(d)        # ← これはユークリッド距離！
```

#### 間違い 2: nprobe や efSearch の調整を怠る（デフォルトでは低 recall）

IVF のデフォルト `nprobe=1`、HNSW のデフォルト `efSearch=16` では、recall が著しく低くなることがあります。必ず評価データで最適値を探しましょう。

```python
# IVF: nprobe を設定
index.nprobe = 32  # デフォルトの1ではなく、適切な値に設定

# HNSW: efSearch を設定
index.hnsw.efSearch = 64  # 精度が必要なら大きく
```

#### 間違い 3: IVF インデックスの train() を忘れる

IVF 系インデックスは `add()` の前に `train()` を呼ぶ必要があります。これを忘れるとエラーになります。Flat や HNSW は `train()` 不要です。

```python
# IVF 系: train → add の順序が必須
index.train(training_data)  # ← これを忘れがち！
index.add(database)

# Flat, HNSW: add のみでOK
index.add(database)
```

### 9.4 自己評価クイズ

---

**Q1.** IVF の `nprobe` を `nlist` と同じ値に設定するとどうなりますか？ その理由も説明してください。

<details>
<summary>回答を表示</summary>

**全探索（Brute Force）と同じ結果** になります。nprobe = nlist は「全クラスタを探索する」ことを意味するため、全データ点との距離を計算します。recall は 1.0 になりますが、IVF のオーバーヘッド分だけ Flat より若干遅くなります。

</details>

---

**Q2.** Product Quantization（PQ）で M=8, nbits=8 とした場合、128次元の float32 ベクトル1つあたりのメモリ使用量は元の何分の1になりますか？

<details>
<summary>回答を表示</summary>

元のサイズ: 128 × 4 byte = **512 byte**
PQ 後のサイズ: M × (nbits/8) = 8 × 1 = **8 byte**
圧縮率: 512 / 8 = **64倍**

（コードブック自体のメモリは M × 2^nbits × (D/M) × 4 byte だが、データ数 N が大きい場合は無視できる）

</details>

---

**Q3.** HNSW で `efSearch` を小さくしすぎるとどのような問題が起きますか？ また、大きくしすぎた場合はどうなりますか？

<details>
<summary>回答を表示</summary>

- **efSearch が小さすぎる場合**: 探索範囲が狭く、局所最適に陥りやすい。真の最近傍を見逃し、**recall が低下** します。
- **efSearch が大きすぎる場合**: 探索範囲が広がりすぎて、多くのノードを訪問する。recall は向上しますが、**検索時間が増加** し、全探索に近づきます。

実運用では、recall と速度のバランスを見て 16〜256 程度に設定します。

</details>

---

**Q4.** FAISS でコサイン類似度に基づく検索を行いたい場合、どのインデックスを使い、データにどのような前処理が必要ですか？

<details>
<summary>回答を表示</summary>

1. データベクトルとクエリベクトルを **L2正規化** する（各ベクトルのノルムを1にする）
   ```python
   faiss.normalize_L2(vectors)
   ```
2. **IndexFlatIP**（内積インデックス）を使う
   ```python
   index = faiss.IndexFlatIP(d)
   ```

正規化済みベクトルの内積はコサイン類似度と等価です: cos(a, b) = a・b / (||a|| ||b||) = a・b（||a||=||b||=1 のとき）

</details>

### 9.5 次のステップ

次の **Notebook 156: 距離学習とファインチューニング** では、タスクに特化した埋め込み空間を学習する方法を学びます。

- Contrastive Loss / Triplet Loss
- 埋め込みモデルのファインチューニング
- ドメイン固有の意味検索の改善

---

## 参考文献

1. Johnson, J., Douze, M., & Jegou, H. (2019). *Billion-scale similarity search with GPUs*. IEEE Transactions on Big Data.
2. Jegou, H., Douze, M., & Schmid, C. (2011). *Product quantization for nearest neighbor search*. IEEE TPAMI.
3. Malkov, Y. A., & Yashunin, D. A. (2018). *Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs*. IEEE TPAMI.
4. FAISS Documentation: https://github.com/facebookresearch/faiss
5. Pinecone. *What is a Vector Database?* https://www.pinecone.io/learn/vector-database/